# Todo App - HTMX v4

Converting main.py to use htmx v4.

In [ ]:
from fasthtml.common import *
from fasthtml.jupyter import *
from hx4_patch.core import *

app,rt,todos,Todo = fast_app(
    'data/todos.db',
    hdrs=[Style(':root { --pico-font-size: 100%; }')],
    id=int, task=str, done=bool, pk='id',
    htmx4=True, htmx=False)

id_curr = 'current-todo'
def tid(id): return f'todo-{id}'

@patch
def __ft__(self:Todo):
    show = AX(self.task, f'/todos/{self.id}', id_curr)
    edit = AX('edit',     f'/edit/{self.id}' , id_curr)
    dt = ' ✅' if self.done else ''
    return Li(show, dt, ' | ', edit, id=tid(self.id))

def mk_input(**kw): return Input(id="new-task", name="task", placeholder="New Todo", required=True, **kw)

@rt("/")
def get():
    add = Form(Group(mk_input(), Button("Add")),
               hx_post="/", target_id='todo-list', hx_swap="beforeend")
    card = Card(Ul(*todos(), id='todo-list'),
                header=add, footer=Div(id=id_curr)),
    title = 'Todo list'
    return Title(title), Main(H1(title), card, cls='container')

@rt("/todos/{id}")
def delete(id:int):
    todos.delete(id)
    return clear(id_curr)

@rt("/")
def post(todo:Todo): return todos.insert(todo), mk_input(hx_swap_oob='true')

@rt("/edit/{id}")
def get(id:int):
    res = Form(Group(Input(id="task"), Button("Save")),
        Hidden(id="id"), CheckboxX(id="done", label='Done'),
        hx_put="/", hx_swap="outerHTML", target_id=tid(id), id="edit")
    return fill_form(res, todos.get(id))

@rt("/")
def put(todo: Todo): return todos.upsert(todo), clear(id_curr)

@rt("/todos/{id}")
def get(id:int):
    todo = todos.get(id)
    btn = Button('delete', hx_delete=f'/todos/{todo.id}',
                 target_id=tid(todo.id), hx_swap="outerHTML")
    return Div(Div(todo.task), btn)

srv = JupyUvi(app)